In [1]:
# Step-by-Step Implementation: Text Classification with BERT
# We will use the transformers library from Hugging Face to implement BERT for text classification.

# Step 1: Install Necessary Libraries
# Install the Hugging Face transformers library and other dependencies.

!pip install transformers datasets torch

   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   --------------------- ------------------ 5.8/10.7 MB 32.0 MB/s eta 0:00:01
   ---------------------------------------  10.5/10.7 MB 34.4 MB/s eta 0:00:01
   ---------------------------------------- 10.7/10.7 MB 23.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   --------------------------------------- 527.0/527.0 kB 16.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   -- ------------------------------------- 6.8/114.6 MB 32.3 MB/s eta 0:00:04
   ---- ----------------------------------- 13.9/114.6 MB 33.6 MB/s eta 0:00:03
   ------- -------------------------------- 21.0/114.6 MB 34.0 MB/s eta 0:00:03
   --------- ------------------------------ 26.5/114.6 MB 32.3 MB/s eta 0:00:03
   ----------- ---------------------------- 33.6/114.6 MB 32.3 MB/s eta 0:00:03
   -------------- ------------------------- 40.6/114.6 MB 32.7 MB/s eta 0

In [2]:
# Step 2: Import Libraries

import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [3]:
# Step 3: Prepare the Dataset

# Example dataset of product reviews
data = [
    ("This product is amazing! Highly recommend.", 1),
    ("Worst purchase ever. Completely useless.", 0),
    ("Fantastic performance. Exceeded my expectations.", 1),
    ("Do not waste your money on this.", 0),
    ("Good value for the price.", 1),
    ("Terrible quality. It broke in a week.", 0),
]

# Split into training and testing sets
texts, labels = zip(*data)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=0.3, random_state=42
)

In [15]:
import torch
from torch. utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
!pip install ipywidgets


# Load tokenizer (recommended instead of BertTokenizer)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Tokenize your text data
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=128
)

# Custom dataset class
class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Create datasets
train_dataset = SentimentDataset(train_encodings, train_labels)
test_dataset = SentimentDataset(test_encodings, test_labels)

# Optional: DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)


In [17]:
!pip install ipywidgets
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [19]:
from torch.optim import AdamW
from torch.utils.data import DataLoader

# Define optimizer and data loader
optimizer = AdamW(model.parameters(), lr=5e-5)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

# Training loop
model.train()
for epoch in range(3):  # 3 epochs
    for batch in train_loader:
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} completed.")


Epoch 1 completed.
Epoch 2 completed.
Epoch 3 completed.


In [21]:
from torch.utils.data import DataLoader
import torch
from sklearn.metrics import accuracy_score, classification_report

# Evaluate the model
test_loader = DataLoader(test_dataset, batch_size=4)
model.eval()

preds = []
with torch.no_grad():
    for batch in test_loader:
        outputs = model(**batch)
        preds.extend(torch.argmax(outputs.logits, axis=1).tolist())

# Print accuracy and classification report
print("Accuracy:", accuracy_score(test_labels, preds))
print(classification_report(test_labels, preds, target_names=["Negative", "Positive"]))


Accuracy: 0.5
              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00         1
    Positive       0.50      1.00      0.67         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2



C:\Users\rumsh\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\rumsh\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\rumsh\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
